# Tutorial 2 Exercise: Data Cleaning and Regression

## Objective
- Apply chemical engineering domain knowledge to clean air quality data
- Use the chemical equation NOx = NO + NO2 to recover missing values
- Build and evaluate regression models
- Compare different train-test splits and model complexities

In [12]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## Section 1: Load Data

In [13]:
df = pd.read_csv('city_day_data.csv')
print('Shape:', df.shape)
df.head()

Shape: (851, 16)


,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Amaravati,2017-11-24,71.36,115.75,1.75,20.65,22.40,12.19,0.10,10.76,109.26,0.17,5.92,NaN,136.366897,Moderate
1,Amaravati,2017-11-25,81.40,124.50,1.44,20.50,21.94,NaN,0.12,15.24,127.09,NaN,6.50,0.06,170.641379,Moderate
2,Amaravati,2017-11-26,78.32,129.06,1.26,26.00,27.26,10.28,0.14,NaN,117.44,0.22,7.95,0.08,160.126897,Moderate
3,Amaravati,2017-11-27,88.76,135.32,6.60,30.85,37.45,12.91,NaN,33.59,111.81,0.29,7.63,0.12,195.766897,Moderate
4,Amaravati,2017-11-28,64.18,104.09,2.56,28.07,30.63,11.42,0.09,19.00,138.18,0.17,5.02,0.07,155.937612,Moderate


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 851 entries, 0 to 850
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   City        851 non-null    object 
 1   Date        851 non-null    object 
 2   PM2.5       772 non-null    float64
 3   PM10        772 non-null    float64
 4   NO          746 non-null    float64
 5   NO2         754 non-null    float64
 6   NOx         731 non-null    float64
 7   NH3         792 non-null    float64
 8   CO          805 non-null    float64
 9   SO2         787 non-null    float64
 10  O3          768 non-null    float64
 11  Benzene     806 non-null    float64
 12  Toluene     770 non-null    float64
 13  Xylene      767 non-null    float64
 14  AQI         851 non-null    float64
 15  AQI_Bucket  851 non-null    object 
dtypes: float64(13), object(3)
memory usage: 106.5+ KB


## Section 2: Apply Chemical Knowledge

**Chemical equation:** NOx = NO + NO2

**Instructions:** When exactly one of these three values is missing, calculate it using the other two.

In [15]:
# Make a copy to work with
df_clean = df.copy()

# Loop through each row
for idx, row in df_clean.iterrows():
    # Check which values are missing
    no_missing = pd.isna(row['NO'])
    no2_missing = pd.isna(row['NO2'])
    nox_missing = pd.isna(row['NOx'])
    
    # Case 1: Only NO is missing
    if no_missing and not no2_missing and not nox_missing:
        # TODO: Calculate NO from NOx and NO2
        df_clean.at[idx, 'NO'] = ___
    
    # Case 2: Only NO2 is missing
    elif no2_missing and not no_missing and not nox_missing:
        # TODO: Calculate NO2 from NOx and NO
        df_clean.at[idx, 'NO2'] = ___
    
    # Case 3: Only NOx is missing
    elif nox_missing and not no_missing and not no2_missing:
        # TODO: Calculate NOx from NO and NO2
        df_clean.at[idx, 'NOx'] = ___

print('Algebra complete!')

ValueError: Incompatible indexer with Series

## Section 3: Basic Data Cleaning

**Instructions:**
1. Remove negative pollutant values (set to NaN)
2. Fix PM violations where PM2.5 > PM10
3. Drop rows with remaining missing values

In [ ]:
# Remove negative pollutant values
pollutants = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3']
for col in pollutants:
    if col in df_clean.columns:
        # TODO: Set negative values to NaN
        df_clean.loc[df_clean[col] < 0, col] = ___

# Fix PM violations: PM2.5 should not exceed PM10
pm_violation = (df_clean['PM2.5'] > df_clean['PM10'])
if pm_violation.sum() > 0:
    # TODO: Set PM2.5 = PM10 * 0.95 for violation rows
    df_clean.loc[pm_violation, 'PM2.5'] = ___

print(f'Before dropna: {len(df_clean)} rows')

# Drop rows with any remaining missing values in pollutants
df_clean = df_clean.dropna(subset=['PM2.5', 'PM10', 'NO', 'NO2', 'NOx'])

print(f'After dropna: {len(df_clean)} rows')

## Section 4: Feature Engineering

**Instructions:** Calculate PM_Ratio = PM2.5 / PM10

In [ ]:
# TODO: Create PM_Ratio feature
df_clean['PM_Ratio'] = ___

# Verify NOx balance (should be close to zero)
nox_error = (df_clean['NOx'] - (df_clean['NO'] + df_clean['NO2'])).abs()
print(f'Max NOx balance error: {nox_error.max():.2f}')

df_clean[['PM2.5', 'PM10', 'PM_Ratio', 'NO', 'NO2', 'NOx']].head(10)

## Section 5: Train-Test Split Comparison

**Instructions:** Test different split ratios and compare their performance.

In [ ]:
# Prepare features and target
X = df_clean[['PM2.5', 'PM_Ratio']]
y = df_clean['PM10']

# Test different split ratios
split_options = [0.1, 0.2, 0.3, 0.4]
results = []

for test_size in split_options:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
    
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    train_r2 = r2_score(y_train, model.predict(X_train))
    test_r2 = r2_score(y_test, model.predict(X_test))
    test_rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
    
    results.append({
        'test_size': test_size,
        'train_r2': round(train_r2, 4),
        'test_r2': round(test_r2, 4),
        'gap': round(train_r2 - test_r2, 4),
        'test_rmse': round(test_rmse, 2)
    })

split_df = pd.DataFrame(results)
split_df

In [ ]:
# Plot the results
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: R2 scores
axes[0].plot(split_df['test_size'], split_df['train_r2'], 'o-', label='Train R2')
axes[0].plot(split_df['test_size'], split_df['test_r2'], 's-', label='Test R2')
axes[0].set_xlabel('Test Size')
axes[0].set_ylabel('R2')
axes[0].set_title('R2 vs Split Ratio')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot 2: Gap
axes[1].bar(split_df['test_size'].astype(str), split_df['gap'])
axes[1].set_xlabel('Test Size')
axes[1].set_ylabel('Train R2 - Test R2')
axes[1].set_title('Generalization Gap')
axes[1].grid(axis='y', alpha=0.3)

# Plot 3: RMSE
axes[2].plot(split_df['test_size'], split_df['test_rmse'], 'D-', color='red')
axes[2].set_xlabel('Test Size')
axes[2].set_ylabel('RMSE')
axes[2].set_title('Test RMSE')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Question:** Which test_size would you choose and why?

In [ ]:
# TODO: Set your chosen test size
best_test_size = ___
print(f'Chosen test_size: {best_test_size}')

## Section 6: Model Comparison

**Instructions:** Compare linear vs polynomial regression.

In [ ]:
# Split data using chosen ratio
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=best_test_size, random_state=42)

# Test different polynomial degrees
degrees = [1, 2, 3]
model_results = []
trained_models = {}

for degree in degrees:
    if degree == 1:
        model = LinearRegression()
        name = 'Linear (degree 1)'
    else:
        model = Pipeline([
            ('poly', PolynomialFeatures(degree=degree)),
            ('linear', LinearRegression())
        ])
        name = f'Polynomial (degree {degree})'
    
    model.fit(X_train, y_train)
    
    train_r2 = r2_score(y_train, model.predict(X_train))
    test_r2 = r2_score(y_test, model.predict(X_test))
    
    model_results.append({
        'model': name,
        'train_r2': round(train_r2, 4),
        'test_r2': round(test_r2, 4),
        'gap': round(train_r2 - test_r2, 4)
    })
    
    trained_models[name] = model

model_df = pd.DataFrame(model_results)
model_df

In [ ]:
# Plot model comparison
x_pos = np.arange(len(model_df))
width = 0.35

plt.figure(figsize=(10, 5))
plt.bar(x_pos - width/2, model_df['train_r2'], width, label='Train R2')
plt.bar(x_pos + width/2, model_df['test_r2'], width, label='Test R2')
plt.xticks(x_pos, model_df['model'])
plt.ylabel('R2')
plt.title('Model Performance Comparison')

# Set y-axis to emphasize differences
all_r2 = pd.concat([model_df['train_r2'], model_df['test_r2']])
plt.ylim(all_r2.min() - 0.02, 1.01)

plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

**Question:** Which model would you choose and why?

In [ ]:
# TODO: Enter your chosen model name (e.g., 'Linear (degree 1)')
best_model_name = '___'
final_model = trained_models[best_model_name]
print(f'Chosen model: {best_model_name}')

## Section 7: Final Evaluation

In [ ]:
# Get predictions
y_pred = final_model.predict(X_test)

# Calculate metrics
final_r2 = r2_score(y_test, y_pred)
final_mae = mean_absolute_error(y_test, y_pred)
final_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'Final Model: {best_model_name}')
print(f'R2:   {final_r2:.4f}')
print(f'MAE:  {final_mae:.2f}')
print(f'RMSE: {final_rmse:.2f}')

In [ ]:
# Plot actual vs predicted
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred, alpha=0.5)
mn, mx = y_test.min(), y_test.max()
plt.plot([mn, mx], [mn, mx], 'r--', lw=2)
plt.xlabel('Actual PM10')
plt.ylabel('Predicted PM10')
plt.title('Actual vs Predicted')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Final Questions

1. Which train:test split did you choose and why?
2. Which model did you choose and why?
3. How did the chemical equation NOx = NO + NO2 help with data cleaning?
4. What does PM_Ratio represent physically?